[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/38_grpo_loss.ipynb)

# 🔴 Hard: GRPO (Group Relative Policy Optimization) Loss

*RLHF & Preference Losses*
Implement the **GRPO** loss.

For each prompt you sample a *group* of responses. The advantage of a response
is its reward standardised **within its own group**:

$$A_i = \frac{r_i - \mu_{g(i)}}{\sigma_{g(i)} + \epsilon},
\qquad \mathcal{L} = -\frac{1}{B}\sum_i \text{sg}[A_i]\,\log \pi(y_i)$$

### Signature
```python
def grpo_loss(logps, rewards, group_ids, eps=1e-5):
    ...  # -> scalar
```

- `logps`: `(B,)` policy log-probability of each sampled response
- `rewards`: `(B,)` scalar reward per response
- `group_ids`: `(B,)` integers — equal ids mean the same prompt

### Rules
- Standardise within each group using the **biased** std (`ddof=0`)
- **Stop the gradient** through the advantages
- Return the mean over the batch, as a scalar

### Why the group replaces the value network
PPO needs a learned value function $V(s)$ to compute advantages, which means a
second network the size of the policy — extra memory, extra training, and a
common source of instability when it lags behind.

GRPO's observation: if you sample $G$ responses to the *same* prompt, the group
mean is already an unbiased baseline for that prompt. Subtracting it removes the
prompt-difficulty confound that the value network existed to model, and dividing
by the group std normalises the scale. So the critic disappears entirely — which
is most of why GRPO is cheaper than PPO and why DeepSeek-R1 used it.

### Why the advantages are detached
$A$ is a *target*, computed from rewards the policy does not differentiate
through. Letting gradient flow into it would optimise the baseline rather than
the policy — the model could lower the loss by manipulating the normalisation
instead of by producing better responses.

### The trap
A group of identical rewards has $\sigma = 0$, so every advantage is 0 and that
group contributes nothing — correct, and the reason $\epsilon$ sits in the
denominator rather than being a division guard you can skip. Getting a `NaN`
here means $\epsilon$ was left out.

### ⚠️ A JAX note
PyTorch writes `advantages[mask] = ...` per group. JAX arrays are immutable and
a boolean mask has data-dependent shape, so build the result with `jnp.where`
over the whole batch, one group at a time.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


def grpo_loss(logps, rewards, group_ids, eps=1e-5):
    """Group Relative Policy Optimization loss.

    Args:
        logps:     (B,) policy log-probs for each sampled response
        rewards:   (B,) scalar reward per response
        group_ids: (B,) integers; equal ids belong to the same prompt
        eps:       stability term in the advantage denominator

    Returns:
        Scalar loss.
    """
    pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax.numpy as jnp

# Two prompts, three sampled responses each.
logps = jnp.array([-1.0, -2.0, -0.5, -1.5, -0.8, -2.2])
rewards = jnp.array([1.0, 0.0, 2.0, 5.0, 4.0, 6.0])   # group 1 is easier
group_ids = jnp.array([0, 0, 0, 1, 1, 1])

print("loss:", float(grpo_loss(logps, rewards, group_ids)))
print("\nGroup 1 has far higher raw rewards, but after within-group")
print("normalisation both groups contribute advantages on the same scale —")
print("that is the prompt-difficulty confound the value network used to model.")

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution

check("grpo_loss")

# hint("grpo_loss")      # stuck? nudge without the answer
# solution("grpo_loss")  # spoiler: the reference implementation